# 04 — ASR (nhận dạng giọng nói) toàn bộ video (resume theo video)

**Phụ trách:** Tuấn Anh (N1) · **Mốc:** M2 (chạy nền tiếp các tuần sau) · **GPU:** CẦN (T4)

Dùng faster-whisper (nhanh, ít VRAM). Kết quả: 1 file jsonl / video, mỗi dòng một đoạn
`{start, end, text}` → N1 gộp vào BM25 theo timestamp keyframe.

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
%pip -q install faster-whisper
import torch; assert torch.cuda.is_available(), 'Chưa bật GPU!'

# === CONFIG ===
DRIVE_ROOT = '/content/drive/MyDrive/AIC2026'
VIDEO_DIR  = f'{DRIVE_ROOT}/data/videos'
OUT_DIR    = f'{DRIVE_ROOT}/outputs/asr'
MODEL_SIZE = 'medium'   # 'large-v3' chính xác hơn nhưng chậm; 'medium' cân bằng tốt
VIDEO_START, VIDEO_END = 0, 99999   # ← MỖI TÀI KHOẢN SỬA DẢI NÀY (chia theo chỉ số video)
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# === ASR từng video, bỏ qua video đã xong ===
from faster_whisper import WhisperModel
import glob, os, json

model = WhisperModel(MODEL_SIZE, device='cuda', compute_type='float16')
videos = sorted(glob.glob(f'{VIDEO_DIR}/*.mp4'))[VIDEO_START:VIDEO_END+1]
print(f'{len(videos)} video trong dải được giao')

for vp in videos:
    video_id = os.path.splitext(os.path.basename(vp))[0]
    out_f = f'{OUT_DIR}/{video_id}.jsonl'
    if os.path.exists(out_f):
        continue
    try:
        segments, info = model.transcribe(vp, language='vi', vad_filter=True)
        with open(out_f + '.tmp', 'w', encoding='utf-8') as f:
            for seg in segments:
                f.write(json.dumps({'start': round(seg.start, 2),
                                    'end': round(seg.end, 2),
                                    'text': seg.text.strip()}, ensure_ascii=False) + '\n')
        os.rename(out_f + '.tmp', out_f)   # atomic: ngắt phiên giữa chừng không tạo file hỏng
        print(f'✓ {video_id}')
    except Exception as e:
        print(f'✗ {video_id}: {e}')
print('XONG dải được giao.')